# 04 — Preprocessing & Feature Engineering

**Owner:** Umer's lane — imputation, scaling, encoding, feature engineering, `src/pipeline.py`.

**Input (handoff from notebook 03):** `data/processed/eligible_features.json` (leakage-safe feature list) and the split decision. Do not proceed until those are populated by notebook 03.

**Output:** a fitted preprocessing pipeline in `src/pipeline.py`, and the fixed train/test split saved to `data/processed/`.

Viva questions to be ready for: Why fit only on train? Why median imputation / RobustScaler? Which engineered features and why?

Remember: every important choice gets a decision-log cell below it (what we decided, evidence, alternative considered, why rejected).

In [1]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))

In [2]:
import json

import numpy as np
import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import RobustScaler
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import StratifiedGroupKFold

from src.config import RANDOM_STATE, PROCESSED_DATA_DIR
from src.pipeline import load_data, build_preprocessing_pipeline

pd.set_option("display.max_columns", None)

In [3]:
df = load_data()
df.shape

(7409, 24)

## Load the eligible-feature list (from notebook 03)

In [4]:
with open(PROCESSED_DATA_DIR / "eligible_features.json") as f:
    eligible_features = json.load(f)

eligible_features

['Current_J0',
 'Current_J1',
 'Current_J2',
 'Current_J3',
 'Current_J4',
 'Current_J5',
 'Temperature_T0',
 'Temperature_J1',
 'Temperature_J2',
 'Temperature_J3',
 'Temperature_J4',
 'Temperature_J5',
 'Tool_current',
 'grip_lost']

## Drop the unrecoverable rows

TODO: per notebook 01, 54 rows have missing values and are also missing the target (46 missing every sensor) — confirm and drop those; they cannot be used for supervised training regardless of imputation strategy.

In [5]:
df = df.dropna(subset=["Robot_ProtectiveStop"]).reset_index(drop=True)
df.shape

(7355, 24)

## Feature engineering

`total_joint_current` (sum of |Current_J0..J5|) — confirmed in notebook 03 as a real, non-leaky signal (rises before and during a stop, unlike speed which is tautological).

`total_joint_current_delta3` — the change in `total_joint_current` over the last 3 readings within the same cycle, operationalizing the "current is trending up before a stop" pattern found in notebook 03's event-window plot. This introduces genuine `NaN`s for the first 3 rows of every cycle (no prior reading to diff against within that cycle) — unlike the raw eligible features, which had zero missing values. This is why the median imputer below is not just a defensive/production step for this column; it does real work here.

This must run before the split/imputation/scaling cells below, since those fit on whatever `eligible_features` contains at that point.

In [6]:
current_cols = [c for c in df.columns if c.startswith("Current_")]
df["total_joint_current"] = df[current_cols].abs().sum(axis=1)

# NaN for the first 3 rows of each cycle (no prior reading within that cycle to diff against)
df["total_joint_current_delta3"] = df["total_joint_current"] - df.groupby("cycle")["total_joint_current"].shift(3)

eligible_features = eligible_features + ["total_joint_current", "total_joint_current_delta3"]
print("NaNs introduced by the lag feature:", df["total_joint_current_delta3"].isna().sum())
eligible_features

NaNs introduced by the lag feature: 720


['Current_J0',
 'Current_J1',
 'Current_J2',
 'Current_J3',
 'Current_J4',
 'Current_J5',
 'Temperature_T0',
 'Temperature_J1',
 'Temperature_J2',
 'Temperature_J3',
 'Temperature_J4',
 'Temperature_J5',
 'Tool_current',
 'grip_lost',
 'total_joint_current',
 'total_joint_current_delta3']

## Apply the split decision (from notebook 03)

TODO: reproduce the exact split notebook 03 decided on (StratifiedGroupKFold-by-cycle holdout, or chronological). Fit everything below on **train only**.

In [7]:
with open(PROCESSED_DATA_DIR / "split_cycles.json") as f:
    split_info = json.load(f)

train_cycles = split_info["train_cycles"]
test_cycles = split_info["test_cycles"]

train_df = df[df["cycle"].isin(train_cycles)].reset_index(drop=True)
test_df = df[df["cycle"].isin(test_cycles)].reset_index(drop=True)

print("train rows:", len(train_df), "| test rows:", len(test_df))

train rows: 5468 | test rows: 1887


## Imputation

TODO: median imputation for sensor columns (robust to outliers, unlike mean). Fit the imputer on train only, then transform both train and test with the fitted imputer.

In [8]:
imputer = SimpleImputer(strategy="median")
imputer.fit(train_df[eligible_features])

SimpleImputer(strategy='median')

## Scaling

TODO: RobustScaler (uses median/IQR, robust to the outliers found in notebook 02). Fit on train only.

In [9]:
continuous_cols = [c for c in eligible_features if c != "grip_lost"]
binary_cols = ["grip_lost"]

scaler = RobustScaler()
scaler.fit(train_df[continuous_cols])

RobustScaler()

## Feature selection results

TODO: any additional selection on top of the eligible-feature list (e.g. dropping near-zero-variance or highly correlated features found in notebook 02).

In [10]:
correlation = train_df[continuous_cols].corr()
correlation["total_joint_current"].sort_values(ascending=False)

total_joint_current           1.000000
total_joint_current_delta3    0.721223
Current_J0                    0.038567
Current_J5                    0.019242
Current_J4                    0.018498
Temperature_J4                0.004058
Temperature_J5                0.003404
Temperature_J3                0.002808
Temperature_J2                0.000663
Temperature_J1               -0.000035
Temperature_T0               -0.000484
Tool_current                 -0.055373
Current_J3                   -0.350582
Current_J2                   -0.589680
Current_J1                   -0.651970
Name: total_joint_current, dtype: float64

## Assemble the preprocessing pipeline

TODO: combine imputer + scaler (+ any feature engineering as a custom transformer) into a single `sklearn` `Pipeline`/`ColumnTransformer`. Once finalized, move the builder function into `src/pipeline.py` (e.g. `build_preprocessing_pipeline()`) so notebooks 05-07 and the backend all reuse the exact same object — per the working agreement, shared code lives in `src/`, never copy-pasted between notebooks.

In [11]:
preprocessing_pipeline = build_preprocessing_pipeline(continuous_cols, binary_cols)
preprocessing_pipeline.fit(train_df[eligible_features])

output_columns = continuous_cols + binary_cols
output_columns

['Current_J0',
 'Current_J1',
 'Current_J2',
 'Current_J3',
 'Current_J4',
 'Current_J5',
 'Temperature_T0',
 'Temperature_J1',
 'Temperature_J2',
 'Temperature_J3',
 'Temperature_J4',
 'Temperature_J5',
 'Tool_current',
 'total_joint_current',
 'total_joint_current_delta3',
 'grip_lost']

## Save the fixed train/test split

TODO: persist the split so every model notebook (05-07) loads the identical rows — never re-split downstream. Keep the `cycle` column alongside the features for the `StratifiedGroupKFold` CV harness in notebook 05.

In [12]:
train_df.to_parquet(PROCESSED_DATA_DIR / "train.parquet", index=False)
test_df.to_parquet(PROCESSED_DATA_DIR / "test.parquet", index=False)
print("saved:", PROCESSED_DATA_DIR / "train.parquet", "|", PROCESSED_DATA_DIR / "test.parquet")

saved: C:\Projects-Uni\UR3-CobotOps-Fault-Detection\data\processed\train.parquet | C:\Projects-Uni\UR3-CobotOps-Fault-Detection\data\processed\test.parquet


## Decision log

### Decision: Median imputation for continuous features, passthrough (no scaling) for grip_lost
- **Evidence:** Notebook 01 found all 54 rows with missing values also had a missing target, so after dropping those, the 14 eligible features have zero missing values in the raw data — the imputer is a defensive step for production (a live sensor reading could arrive incomplete), not a fix for a real gap here. `grip_lost` is a rare (~3.3%) binary flag; `RobustScaler`'s median/IQR both land on 0 for it, so scaling it would either silently no-op (via sklearn's internal IQR=0 guard) or be numerically meaningless. We explicitly exclude it from scaling via a `ColumnTransformer` passthrough branch instead of relying on that edge case.
- **Alternative considered:** Scale `grip_lost` along with everything else (Option A).
- **Why rejected:** Relying on an unstated library edge case (IQR=0 fallback) instead of an explicit design choice is harder to defend and reason about; a binary flag doesn't have a meaningful "scaled" interpretation.

### Decision: Engineer total_joint_current and total_joint_current_delta3
- **Evidence:** Notebook 03's event-window study found `total_joint_current` rises both concurrently with and in the rows *before* a stop — unlike speed, this isn't tautological. `total_joint_current_delta3` (change over the last 3 readings within the same cycle) operationalizes that pre-stop rising-trend pattern directly, rather than relying on the model to infer it from raw same-row values.
- **Alternative considered:** Use only the raw per-joint Current_J0-J5 columns without an aggregate or trend feature.
- **Why rejected:** The raw columns don't directly encode "is current trending up," which is the actual pattern found in notebook 03; the engineered features make that signal explicit. Note: `total_joint_current_delta3` introduces 720 genuine NaNs (3 per cycle × 240 cycles, since the first 3 rows of every cycle have no prior reading to diff against) — this is the one place the median imputer does real work, not just defensive work.

### Decision: Keep all 16 features, no drops from correlation check
- **Evidence:** `total_joint_current` correlates with `Current_J1` (-0.65) and `Current_J2` (-0.59) — negative because the aggregate sums absolute values while the raw columns are signed and run consistently negative (medians around -2.2 and -1.1), so larger-magnitude (more negative) raw current corresponds to a larger absolute-value sum. `Current_J0/J4/J5` barely correlate with it (0.02-0.04) since their raw magnitudes sit near zero. No pair exceeds |r|=0.65.
- **Alternative considered:** Drop `Current_J1`/`Current_J2` as redundant with `total_joint_current`, or drop `total_joint_current` as redundant with the raw columns.
- **Why rejected:** |r|=0.65 is moderate, not the >0.9 territory that would indicate true redundancy; the raw signed columns preserve directional information the absolute-value aggregate destroys, and tree-based models (XGBoost, Random Forest — two of the four planned algorithms) handle correlated features without the instability linear models can have, so there's no strong reason to drop anything at this feature count.